# Zinc certificate valuation
## Primary result: certificate vs physical

The objective is to measure the certificate premium or discount to the domestic physical benchmark.
The main chart uses the approved bounded, interpolated physical-to-intrinsic ratio method:
**100 × (certificate price / estimated physical price − 1)**.
Dots identify observed physical anchors; the remaining eligible dates use the approved interpolation.
Positive values indicate a premium; negative values indicate a discount.

Sources: IME certificate and physical transactions, Westmetall LME quotations, and the shared free-market USD/IRR series.
This presentation reads existing processed outputs; refresh the project before reviewing it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'shared').is_dir() and (p / 'commodity/zinc').is_dir())
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
from shared.notebook_tools.commodity_dashboard import valuation_figures, PLOTLY_CONFIG
PROJECT = WORKSPACE / 'commodity/zinc'
approved_primary = pd.read_csv(PROJECT / 'data/processed/bubble/zinc_certificate_bubble.csv', parse_dates=['date'])
figures = valuation_figures(PROJECT, 'Zinc')
figures[0].show(config=PLOTLY_CONFIG)
values = approved_primary['certificate_bubble_pct']
display(pd.DataFrame([{'Observations': len(values), 'From': approved_primary['date'].min(),
    'To': approved_primary['date'].max(), 'Mean (%)': values.mean(),
    'Median (%)': values.median(), 'Latest (%)': approved_primary.sort_values('date')['certificate_bubble_pct'].iloc[-1]}]))


## Supporting result: certificate vs intrinsic

Intrinsic reference = LME cash USD/kg × USD/IRR. This is a separate comparison, not the primary certificate-to-physical bubble.

In [ ]:
figures[1].show(config=PLOTLY_CONFIG)

## Supporting result: physical vs intrinsic

This chart measures the domestic physical-market premium or discount to the same international reference.

In [ ]:
figures[2].show(config=PLOTLY_CONFIG)

## Reading the results

Use the first chart for the main certificate-to-physical comparison. The next two charts provide international-value context and are not interchangeable with it. Statistics above are calculated from the loaded data rather than copied into prose.

---
# Research appendix — supporting work and experimental methods

The following work records how the benchmark and alternative methods were investigated. Experimental regressions do not replace the approved primary result above.

## Benchmark construction and diagnostics

These supporting diagnostics read processed datasets. Regression remains experimental.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

def find_project() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / 'data/processed/bubble/zinc_certificate_bubble.csv').exists():
            return base
        candidate = base / 'commodity/zinc'
        if (candidate / 'data/processed/bubble/zinc_certificate_bubble.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate commodity/zinc')

PROJECT = find_project()
P = PROJECT / 'data/processed'
benchmark = pd.read_csv(P / 'physical' / 'zinc_9798_cash_daily.csv')
physical_direct = pd.read_csv(P / 'bubble' / 'physical_vs_intrinsic_bubble.csv', parse_dates=['date'])
certificate_direct = pd.read_csv(P / 'bubble' / 'certificate_vs_intrinsic_bubble.csv', parse_dates=['date'])
primary = pd.read_csv(P / 'bubble' / 'zinc_certificate_bubble.csv', parse_dates=['date'])
regression = pd.read_csv(P / 'bubble' / 'intrinsic_regression.csv', parse_dates=['date'])
regression_metrics = pd.read_csv(P / 'bubble' / 'intrinsic_regression_metrics.csv')
print(PROJECT)

## Data summary and three bubble definitions

In [ ]:
def describe_bubble(frame, column, label):
    values = frame[column]
    return {
        'definition': label, 'rows': len(frame),
        'first_date': frame['date'].min().date(), 'last_date': frame['date'].max().date(),
        'mean_pct': values.mean(), 'median_pct': values.median(),
        'min_pct': values.min(), 'max_pct': values.max(),
        'positive_days': values.gt(0).sum(), 'negative_days': values.lt(0).sum(),
    }

summary = pd.DataFrame([
    describe_bubble(physical_direct, 'physical_vs_intrinsic_bubble_pct', 'physical / intrinsic'),
    describe_bubble(certificate_direct, 'certificate_vs_intrinsic_bubble_pct', 'certificate / intrinsic'),
    describe_bubble(primary, 'certificate_bubble_pct', 'certificate / estimated physical (primary)'),
])
display(summary.round({'mean_pct': 2, 'median_pct': 2, 'min_pct': 2, 'max_pct': 2}))
display(pd.DataFrame({
    'metric': ['benchmark days','certificate days','exact anchors','interpolated primary days'],
    'value': [len(benchmark), len(certificate_direct), primary['physical_ratio_method'].eq('observed').sum(), primary['physical_ratio_method'].eq('linear_interpolation').sum()]
}))

### Comparison definitions

The three approved comparisons are shown separately at the start of this notebook.

In [ ]:
# Independent approved comparison charts are displayed above.

## Intrinsic price components and physical benchmark

In [ ]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1,
    subplot_titles=('99.97 + 99.98 physical vs LME-FX intrinsic',
                    'Certificate settlement vs LME-FX intrinsic'))
for row, frame, price_column in [(1, physical_direct, 'physical_price_irr_per_kg'),
                                  (2, certificate_direct, 'certificate_price_irr_per_kg')]:
    fig.add_trace(go.Scatter(x=frame['date'], y=frame[price_column],
        name='Observed price', line_color='#1976D2', showlegend=row == 1), row=row, col=1)
    fig.add_trace(go.Scatter(x=frame['date'], y=frame['intrinsic_price_irr_per_kg'],
        name='Intrinsic', line_color='#EF6C00', showlegend=row == 1), row=row, col=1)
    fig.update_yaxes(title_text='IRR/kg', row=row, col=1)
fig.update_layout(height=750, template='plotly_white', hovermode='x unified')
fig.show()
display(benchmark[['physical_trade_date_jalali','grades','grade_99_97_quantity',
    'grade_99_98_quantity','total_quantity','grade_99_97_weighted_price',
    'grade_99_98_weighted_price','physical_weighted_price']].tail(30))

## anchors and interpolation of the main method

In [ ]:
observed = primary['physical_ratio_method'].eq('observed')
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1,
    subplot_titles=('Observed anchors and interpolated ratio',
                    'Certificate settlement and estimated physical price'))
fig.add_trace(go.Scatter(x=primary['date'], y=primary['physical_ratio'],
    name='Physical / intrinsic ratio', line_color='#7B1FA2'), row=1, col=1)
fig.add_trace(go.Scatter(x=primary.loc[observed, 'date'],
    y=primary.loc[observed, 'physical_ratio'], mode='markers',
    name='Exact anchors', marker_color='#263238'), row=1, col=1)
for column, name, color in [
    ('certificate_price_irr_per_kg', 'Certificate settlement', '#1976D2'),
    ('estimated_physical_price_irr_per_kg', 'Estimated physical', '#EF6C00'),
]:
    fig.add_trace(go.Scatter(x=primary['date'], y=primary[column],
        name=name, line_color=color), row=2, col=1)
fig.add_trace(go.Scatter(x=primary.loc[observed, 'date'],
    y=primary.loc[observed, 'observed_physical_price_irr_per_kg'],
    mode='markers', name='Observed physical', marker_color='#263238'), row=2, col=1)
fig.update_yaxes(title_text='IRR/kg', row=2, col=1)
fig.update_layout(height=750, template='plotly_white', hovermode='x unified')
fig.show()
display(primary.loc[observed, ['date','certificate_price_irr_per_kg',
    'observed_physical_price_irr_per_kg','intrinsic_price_irr_per_kg',
    'certificate_bubble_pct']])

## Experimental method of regression and data age control
Regression is not a substitute for the original method. Model selection is done only with TimeSeriesSplit and the lowest out-of-sample RMSE.

In [ ]:
display(regression_metrics.sort_values('timeseries_cv_rmse'))
age = certificate_direct[['date', 'lme_age_days', 'usd_age_days']].set_index('date')
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1,
    subplot_titles=('Primary vs experimental regression bubble',
                    'As-of source age on certificate dates'))
fig.add_trace(go.Scatter(x=primary['date'], y=primary['certificate_bubble_pct'],
    name='Primary interpolation', line_color='#1976D2'), row=1, col=1)
fig.add_trace(go.Scatter(x=regression['date'], y=regression['certificate_bubble_pct'],
    name='Experimental regression', line_color='#7B1FA2'), row=1, col=1)
fig.add_hline(y=0, line_color='#455A64', row=1, col=1)
for column, color in [('lme_age_days', '#EF6C00'), ('usd_age_days', '#00897B')]:
    fig.add_trace(go.Scatter(x=age.index, y=age[column], name=column,
        line_shape='hv', line_color=color), row=2, col=1)
fig.update_yaxes(title_text='Bubble (%)', row=1, col=1)
fig.update_yaxes(title_text='Days old', row=2, col=1)
fig.update_layout(height=750, template='plotly_white', hovermode='x unified')
fig.show()
display(age.describe().T)

## Standard market dashboard

This governed, read-only section uses the same presentation contract across commodity projects:
source coverage, physical and certificate activity, separate price panels, physical-goods
composition, and validated processed bubbles. It never writes raw data or constructs a missing
bubble. For Zinc, product comparability still follows the project-specific workflow.

In [ ]:
from pathlib import Path
import sys

def locate_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "commodity" / "zinc").exists() and (candidate / "shared").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

WORKSPACE_ROOT = locate_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from shared.notebook_tools.commodity_dashboard import (
    goods_type_counts,
    load_markets,
    market_summary,
    plot_available_bubbles,
    plot_goods_type_counts,
    plot_market_prices,
    plot_trade_activity,
)

PROJECT_DIR = WORKSPACE_ROOT / "commodity" / "zinc"
physical_dashboard, certificate_dashboard = load_markets(
    PROJECT_DIR, "zinc", physical_filename=None
)
display(market_summary(physical_dashboard, certificate_dashboard))
plot_trade_activity(physical_dashboard, certificate_dashboard, "Zinc")
plot_market_prices(physical_dashboard, certificate_dashboard, "Zinc")
goods_count_table = plot_goods_type_counts(physical_dashboard, "Zinc", top_n=30)
display(goods_count_table)
# Approved comparison charts are displayed at the start of this notebook.

## Historical bubble distribution

This section reads the standardized processed table and renders an interactive Plotly figure for
each bubble type. The panels show the observed distribution, empirical cumulative distribution
function F(x), and magnitude frequency P(|Bubble| >= |x|). Negative bubbles retain their sign in
the first two panels; the third panel measures magnitude only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def locate_distribution_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "shared").exists() and (candidate / "commodity/zinc").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

distribution_workspace = locate_distribution_workspace()
if str(distribution_workspace) not in sys.path:
    sys.path.insert(0, str(distribution_workspace))

from shared.market_analysis.bubble_distribution import plot_distribution_plotly

distribution_project = distribution_workspace / "commodity/zinc"
distribution_files = list(
    (distribution_project / "data/processed/bubble").glob("*_bubble_distribution.csv")
)
if len(distribution_files) != 1:
    raise ValueError(f"Expected one named bubble distribution CSV, found {distribution_files}")
bubble_distribution = pd.read_csv(distribution_files[0], parse_dates=["observation_date"])
for series_id, series_distribution in bubble_distribution.groupby("series_id", sort=True):
    comparison = series_distribution["comparison"].iloc[0]
    figure = plot_distribution_plotly(series_distribution, comparison)
    figure.show()

display(bubble_distribution)